#ELEC5622 Project 1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/5622project1')

TRAIN_CSV = DATA_DIR / 'AAL_statistics_volumn_train_first90_with_labels.csv'
TEST_CSV  = DATA_DIR / 'AAL_statistics_volumn_test_first90.csv'

#Initialization

In [ ]:
import pandas as pd

train = pd.read_csv(TRAIN_CSV, header=None)
test = pd.read_csv(TEST_CSV, header=None)

roi_cols = [f'ROI_{i}' for i in range(1, 91)]
train.columns = ['subject_id'] + roi_cols + ['label']
test.columns  = ['subject_id'] + roi_cols
print('Train:', train.shape, 'Test:', test.shape)
train.head(3)

In [ ]:
test.head(3)

#Mapping Labels

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
X_train = train[roi_cols].values
y_raw   = train['label']
X_test  = test[roi_cols].values

# mapping labels (AD → 1, NC → 0)
if y_raw.dtype == object:
    y_train = y_raw.map({'AD':1, 'NC':0, 'ad':1, 'nc':0}).astype(int).values
else:
    y_train = y_raw.astype(int).values

print('Class distribution:', dict(zip(*np.unique(y_train, return_counts=True))))
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print(y_train[:40])

#View the number of samples in the two different categories.

Check the number of samples in each category to determine whether the class_weight parameter needs to use the 'balanced' option to adjust for class imbalance.

In [ ]:
import numpy as np

unique, counts = np.unique(y_train, return_counts=True)
total = len(y_train)
for u, c in zip(unique, counts):
    print(f"Class {u}: {c} samples ({c/total:.1%})")

Since the numbers of AD and NC samples are relatively balanced, the class_weight parameter does not need to be set to "balanced".

#Validate whether to use a linear or RBF kernel + normalization (standardization)

Since we don’t know whether the data is linearly separable, both the linear and RBF kernels are defined here.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pipe_linear = Pipeline([("scaler", StandardScaler()),
                        ("svc", SVC(kernel="linear", random_state=42))])

pipe_rbf = Pipeline([("scaler", StandardScaler()),
                        ("svc", SVC(kernel="rbf", gamma="scale", random_state=42))])

#Perform 5-fold stratified cross-validation and model training for each kernel
Quantify the model’s generalization ability. Since the dataset is small (only 40 samples), cross-validation is necessary to improve the SVM’s generalization performance and prevent overfitting.

Perform 5-fold stratified cross-validation for the two different kernels separately to evaluate the AUC performance under each kernel.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_lin = cross_val_score(pipe_linear, X_train, y_train, cv=cv, scoring="roc_auc").mean()#linear核
auc_rbf = cross_val_score(pipe_rbf,   X_train, y_train, cv=cv, scoring="roc_auc").mean()#rbf核
print(f"Linear 5-fold AUC: {auc_lin:.4f}")
print(f"RBF    5-fold AUC: {auc_rbf:.4f}")

The following function shows that both kernels achieve a perfect AUC score during 5-fold cross-validation, indicating that the choice of kernel no longer affects the CV score. Since the AUC score in every fold is 1, this means the model can perfectly and easily distinguish between the AD and NC categories during the stratified 5-fold cross-validation.

This suggests that the data is linearly separable. However, because the sample size is small (only 40 samples), each fold’s training set likely excludes only a few samples, and the model may have memorized noise or specific features. Therefore, this result only shows that the SVM can achieve linear separability in high dimensions within each training fold, but it does not necessarily indicate good generalization performance.

For the linear kernel, each fold achieves an AUC of 1, indicating that the model can perfectly separate the samples in every fold.
For the RBF kernel, the average AUC is 0.9875, suggesting that the RBF kernel introduces slight nonlinear fluctuations after mapping to higher dimensions, but it does not provide any substantial improvement in performance.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Train the model multiple times with different parameter combinations,
#compute the average score for each combination, automatically select the best parameters,
#and then retrain on the entire training set.
#lienar kernel
grid_lin = GridSearchCV(pipe_linear, {"svc__C":[0.001,0.01,0.1,0.3,1,3,10]},
                        scoring="roc_auc", cv=cv, n_jobs=-1, refit=True, verbose=1)

grid_lin.fit(X_train, y_train)#Train on the training dataset using a linear kernel.
print("Linear best:", grid_lin.best_params_, round(grid_lin.best_score_,4))

#rbf kernel
grid_rbf = GridSearchCV(pipe_rbf,
                        {"svc__C":[0.001,0.01,0.1,0.3,1,3,10],
                         "svc__gamma":["scale",1e-4,3e-4,1e-3,3e-3]},
                        scoring="roc_auc", cv=cv, n_jobs=-1, refit=True, verbose=1)
grid_rbf.fit(X_train, y_train)#Train on the training dataset using a rbf kernel.
print("RBF best:", grid_rbf.best_params_, round(grid_rbf.best_score_,4))


def per_fold_auc(estimator, X, y, cv):
    fold_aucs = []
    for tr, va in cv.split(X, y):
        m = clone(estimator).fit(X[tr], y[tr])
        s = m.decision_function(X[va])
        if roc_auc_score(y[va], s) < 0.5:
            s = -s
        fold_aucs.append(roc_auc_score(y[va], s))
    return np.array(fold_aucs)

#Calculate the AUC for each fold under each kernel separately.
auc_lin = per_fold_auc(pipe_linear, X_train, y_train, cv)
auc_rbf = per_fold_auc(pipe_rbf,   X_train, y_train, cv)

#output
print("Linear kernel - per-fold AUC:", np.round(auc_lin, 4),
      "| mean:", round(auc_lin.mean(), 4), "std:", round(auc_lin.std(), 4))

print("RBF kernel    - per-fold AUC:", np.round(auc_rbf, 4),
      "| mean:", round(auc_rbf.mean(), 4), "std:", round(auc_rbf.std(), 4))

#Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Use PCA to project the features into 2D space,
#making it easier to visualize the overall data structure.
pca = PCA(n_components=2, random_state=42)
X2d = pca.fit_transform(X_train)

# define the function for visualization
def plot_decision(ax, model, X, y, title):
    model.fit(X, y)

    x_min, x_max = X[:,0].min() - 0.5, X[:,0].max() + 0.5
    y_min, y_max = X[:,1].min() - 0.5, X[:,1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                         np.linspace(y_min, y_max, 400))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    cs = ax.contourf(xx, yy, Z, levels=30, cmap="coolwarm", alpha=0.35)
    ax.contour(xx, yy, Z, levels=[0], colors="k", linewidths=1.2)

    ax.scatter(X[y==0,0], X[y==0,1], c="#1f77b4", edgecolor="k", label="Class 0", s=50)
    ax.scatter(X[y==1,0], X[y==1,1], c="#d62728", edgecolor="k", label="Class 1", s=50)

    svc = model.named_steps["svc"]
    if hasattr(svc, "support_"):
        sv = model.named_steps["scaler"].transform(X)[svc.support_]
        ax.scatter(X[svc.support_,0], X[svc.support_,1],
                   facecolors="none", edgecolors="k", s=120, linewidths=1.2, label="Support Vectors")

    ax.set_title(title)
    ax.legend(loc="best")

#Calculate the cross-validation scores for different kernels.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_lin = cross_val_score(pipe_linear, X2d, y_train, cv=cv, scoring="roc_auc").mean()
auc_rbf = cross_val_score(pipe_rbf,   X2d, y_train, cv=cv, scoring="roc_auc").mean()

# visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
plot_decision(axes[0], pipe_linear, X2d, y_train, f"Linear SVM (5-fold AUC={auc_lin:.3f})")
plot_decision(axes[1], pipe_rbf,   X2d, y_train, f"RBF SVM (5-fold AUC={auc_rbf:.3f})")
plt.show()

#Learn the optimal threshold.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.base import clone
import numpy as np

#Extract the previously trained model that already contains the learned parameters.
best_lin = grid_lin.best_estimator_
best_rbf = grid_rbf.best_estimator_

def oof_auc_decision(model, X, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_scores = np.zeros(len(y))
    for tr, va in skf.split(X, y):
        m = clone(model).fit(X[tr], y[tr])
        s = m.decision_function(X[va])
        if roc_auc_score(y[va], s) < 0.5:
            s = -s
        oof_scores[va] = s
    fpr, tpr, thr = roc_curve(y, oof_scores)
    return float(thr[(tpr - fpr).argmax()]), oof_scores

#Extract the best threshold parameter.
best_thr_lin, _ = oof_auc_decision(best_lin, X_train, y_train)
best_thr_rbf, _ = oof_auc_decision(best_rbf, X_train, y_train)

#Prediction results for different kernels.

In [ ]:

#Split the test set using the learned threshold.
scores_lin = best_lin.decision_function(X_test)
scores_rbf = best_rbf.decision_function(X_test)

pred_lin = (scores_lin >= best_thr_lin).astype(int)
pred_rbf = (scores_rbf >= best_thr_rbf).astype(int)

print("Linear counts:", (pred_lin==0).sum(), (pred_lin==1).sum())
print("RBF    counts:", (pred_rbf==0).sum(), (pred_rbf==1).sum())

#Merge the test set and the training set, then generate the complete CSV file.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

train_raw = pd.read_csv(TRAIN_CSV, header=None, dtype=str).reset_index(drop=True)
test_raw  = pd.read_csv(TEST_CSV,  header=None, dtype=str).reset_index(drop=True)

n_train_cols = train_raw.shape[1]
n_test_cols  = test_raw.shape[1]
if n_train_cols != n_test_cols + 1:
    raise ValueError(f"training columns ({n_train_cols}) != test columns ({n_test_cols}) + 1，please check.")

pred_test = pred_lin
label_vals = list(pd.unique(train_raw.iloc[:, -1]))
if len(label_vals) != 2:
    raise ValueError(f"the last columns is not binary classifed ={label_vals}")

int2label_str = {0: label_vals[0], 1: label_vals[1]}
test_label_str = pd.Series(np.asarray(pred_test), index=test_raw.index).map(int2label_str).astype(str)


test_labeled = test_raw.copy()
test_labeled[n_test_cols] = test_label_str


test_labeled.columns = range(n_train_cols)

combined_raw = pd.concat([train_raw, test_labeled], ignore_index=True)
OUT_PATH = DATA_DIR / "AAL_statistics_volumn_completed.csv"
combined_raw.to_csv(OUT_PATH, index=False, header=False)
print(f"✅ saved to：{OUT_PATH}")